In [2]:
import numpy as np

# Define probabilities
P_hosteler = 0.60
P_day_scholar = 0.40
P_A_given_hosteler = 0.30
P_A_given_day_scholar = 0.20

# Calculate P(A) using the law of total probability
P_A = (P_hosteler * P_A_given_hosteler) + (P_day_scholar * P_A_given_day_scholar)

# Calculate P(hosteler | A) using Bayes' Theorem
P_hosteler_given_A = (P_hosteler * P_A_given_hosteler) / P_A

print("Probability that the student is a hosteler given they have an A grade:", P_hosteler_given_A)

# Define probabilities
P_disease = 0.01
sensitivity = 0.99
specificity = 0.98

# Probability of a false positive
false_positive_rate = 1 - specificity

# Calculate P(Positive | Disease) and P(Positive | No Disease)
P_positive_given_disease = sensitivity
P_positive_given_no_disease = false_positive_rate

# Calculate P(Disease | Positive) using Bayes' Theorem
P_disease_given_positive = (P_disease * P_positive_given_disease) / ((P_disease * P_positive_given_disease) + ((1
- P_disease) * P_positive_given_no_disease))

print("Probability of having the disease given a positive test result:", P_disease_given_positive)

Probability that the student is a hosteler given they have an A grade: 0.6923076923076923
Probability of having the disease given a positive test result: 0.33333333333333315


In [8]:
import math
from collections import defaultdict

# ============================================================
# HARDCODED DATASET (no CSV needed)
# ============================================================
data = [
    {"age": "<=30",    "income": "high",   "student": "no",  "credit_rating": "fair",      "buys_computer": "no"},
    {"age": "<=30",    "income": "high",   "student": "no",  "credit_rating": "excellent", "buys_computer": "no"},
    {"age": "31...40", "income": "high",   "student": "no",  "credit_rating": "fair",      "buys_computer": "yes"},
    {"age": ">40",     "income": "medium", "student": "no",  "credit_rating": "fair",      "buys_computer": "yes"},
    {"age": ">40",     "income": "low",    "student": "yes", "credit_rating": "fair",      "buys_computer": "yes"},
    {"age": ">40",     "income": "low",    "student": "yes", "credit_rating": "excellent", "buys_computer": "no"},
    {"age": "31...40", "income": "low",    "student": "yes", "credit_rating": "excellent", "buys_computer": "yes"},
    {"age": "<=30",    "income": "medium", "student": "no",  "credit_rating": "fair",      "buys_computer": "no"},
    {"age": "<=30",    "income": "low",    "student": "yes", "credit_rating": "fair",      "buys_computer": "yes"},
    {"age": ">40",     "income": "medium", "student": "yes", "credit_rating": "fair",      "buys_computer": "yes"},
    {"age": "<=30",    "income": "medium", "student": "yes", "credit_rating": "excellent", "buys_computer": "yes"},
    {"age": "31...40", "income": "medium", "student": "no",  "credit_rating": "excellent", "buys_computer": "yes"},
    {"age": "31...40", "income": "high",   "student": "yes", "credit_rating": "fair",      "buys_computer": "yes"},
    {"age": ">40",     "income": "medium", "student": "no",  "credit_rating": "excellent", "buys_computer": "no"},
]


# NAIVE BAYES CLASSIFIER — FROM SCRATCH

class NaiveBayes:
    def __init__(self):
        self.class_priors = {}
        self.feature_probs = {}
        self.classes = []
        self.feature_names = []
        self.feature_values = {}

    def fit(self, data, target_col):
        self.feature_names = [c for c in data[0].keys() if c != target_col]
        n = len(data)

        # Count class occurrences
        class_counts = defaultdict(int)
        for row in data:
            class_counts[row[target_col]] += 1

        self.classes = list(class_counts.keys())

        # Priors P(class)
        for c in self.classes:
            self.class_priors[c] = class_counts[c] / n

        # Possible values per feature
        for f in self.feature_names:
            self.feature_values[f] = set(row[f] for row in data)

        # Likelihoods P(feature_value | class) with Laplace smoothing
        for c in self.classes:
            self.feature_probs[c] = {}
            class_rows = [row for row in data if row[target_col] == c]
            n_c = len(class_rows)
            for f in self.feature_names:
                self.feature_probs[c][f] = {}
                for val in self.feature_values[f]:
                    count = sum(1 for row in class_rows if row[f] == val)
                    # Laplace smoothing
                    self.feature_probs[c][f][val] = (count + 1) / (n_c + len(self.feature_values[f]))

    def predict(self, sample):
        posteriors = {}
        for c in self.classes:
            log_prob = math.log(self.class_priors[c])
            for f in self.feature_names:
                val = sample.get(f)
                prob = self.feature_probs[c][f].get(val, 1e-6)
                log_prob += math.log(prob)
            posteriors[c] = log_prob
        return max(posteriors, key=posteriors.get), posteriors

    def evaluate(self, test_data, target_col):
        tp = fp = tn = fn = 0
        correct = 0
        for row in test_data:
            actual = row[target_col]
            predicted, _ = self.predict(row)
            if predicted == actual:
                correct += 1
            if actual == "yes" and predicted == "yes":
                tp += 1
            elif actual == "no" and predicted == "yes":
                fp += 1
            elif actual == "no" and predicted == "no":
                tn += 1
            elif actual == "yes" and predicted == "no":
                fn += 1

        accuracy = correct / len(test_data)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        return accuracy, precision, recall, f1, (tp, fp, tn, fn)

# RUN

model = NaiveBayes()
model.fit(data, target_col="buys_computer")

# Classify a new sample
new_sample = {"age": "<=30", "income": "medium", "student": "yes", "credit_rating": "fair"}
prediction, posteriors = model.predict(new_sample)

print(" " * 60)
print("NAIVE BAYES — BUY COMPUTER CLASSIFIER")
print(" " * 60)
print(f"\nNew sample: {new_sample}")
print(f"\nPredicted class: {prediction}")
print(f"\nPosterior (log) scores:")
for c, score in posteriors.items():
    print(f"  {c}: {score:.4f}")

# Evaluate on training data
acc, prec, rec, f1, cm = model.evaluate(data, "buys_computer")
print(f"\n--- Evaluation on Training Data ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"Confusion (TP, FP, TN, FN): {cm}")

                                                            
NAIVE BAYES — BUY COMPUTER CLASSIFIER
                                                            

New sample: {'age': '<=30', 'income': 'medium', 'student': 'yes', 'credit_rating': 'fair'}

Predicted class: yes

Posterior (log) scores:
  no: -4.8037
  yes: -3.6076

--- Evaluation on Training Data ---
Accuracy : 0.9286
Precision: 0.9000
Recall   : 1.0000
F1-Score : 0.9474
Confusion (TP, FP, TN, FN): (9, 1, 4, 0)


In [11]:
import math
from collections import defaultdict

# ============================================================
# HARDCODED DATASET (no CSV needed)
# ============================================================
data = [
    {"text": "A great game",                  "tag": "Sports"},
    {"text": "The election was over",         "tag": "Not sports"},
    {"text": "Very clean match",              "tag": "Sports"},
    {"text": "A clean but forgettable game",  "tag": "Sports"},
    {"text": "It was a close election",       "tag": "Not sports"},
]

# ============================================================
# NAIVE BAYES FOR TEXT — FROM SCRATCH
# ============================================================
class NaiveBayesText:
    def __init__(self):
        self.class_priors = {}
        self.word_probs = {}
        self.vocab = set()
        self.classes = []
        self.class_word_counts = {}
        self.class_total_words = {}

    def tokenize(self, text):
        # Lowercase, remove punctuation, split
        text = text.lower()
        for ch in [".", ",", "!", "?", ";", ":"]:
            text = text.replace(ch, "")
        return text.split()

    def fit(self, data):
        n = len(data)
        class_counts = defaultdict(int)

        for row in data:
            c = row["tag"]
            class_counts[c] += 1
            words = self.tokenize(row["text"])
            if c not in self.class_word_counts:
                self.class_word_counts[c] = defaultdict(int)
                self.class_total_words[c] = 0
            for w in words:
                self.class_word_counts[c][w] += 1
                self.class_total_words[c] += 1
                self.vocab.add(w)

        self.classes = list(class_counts.keys())

        # Priors
        for c in self.classes:
            self.class_priors[c] = class_counts[c] / n

        # Likelihoods with Laplace smoothing
        V = len(self.vocab)
        for c in self.classes:
            self.word_probs[c] = {}
            for w in self.vocab:
                count = self.class_word_counts[c].get(w, 0)
                self.word_probs[c][w] = (count + 1) / (self.class_total_words[c] + V)

    def predict(self, text):
        words = self.tokenize(text)
        scores = {}
        V = len(self.vocab)
        for c in self.classes:
            score = math.log(self.class_priors[c])
            for w in words:
                if w in self.vocab:
                    score += math.log(self.word_probs[c][w])
                else:
                    # Unknown word → Laplace-smoothed
                    score += math.log(1 / (self.class_total_words[c] + V))
            scores[c] = score
        return max(scores, key=scores.get), scores

    def evaluate(self, data):
        correct = 0
        tp = fp = tn = fn = 0
        for row in data:
            actual = row["tag"]
            predicted, _ = self.predict(row["text"])
            if predicted == actual:
                correct += 1
            # Treat "Sports" as positive
            if actual == "Sports" and predicted == "Sports":
                tp += 1
            elif actual != "Sports" and predicted == "Sports":
                fp += 1
            elif actual != "Sports" and predicted != "Sports":
                tn += 1
            elif actual == "Sports" and predicted != "Sports":
                fn += 1

        accuracy = correct / len(data)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
        return accuracy, precision, recall, f1


# ============================================================
# RUN
# ============================================================
model = NaiveBayesText()
model.fit(data)

# The target sentence from the question
test_sentence = "A very close game"
prediction, scores = model.predict(test_sentence)

print("-" * 60)
print("NAIVE BAYES — SPORTS TEXT CLASSIFIER")
print(" " * 60)
print(f"\nTest sentence: '{test_sentence}'")
print(f"\nPredicted tag: {prediction}")
print(f"\nScores (log):")
for c, s in scores.items():
    print(f"  {c}: {s:.4f}")

# Evaluate on training data
acc, prec, rec, f1 = model.evaluate(data)
print(f"\n--- Evaluation on Training Data ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")

# Show the math for the target sentence
print("\n--- Manual Calculation for 'A very close game' ---")
words = model.tokenize(test_sentence)
print(f"Tokens: {words}")
print(f"\nP(Sports)     = {model.class_priors.get('Sports', 0):.4f}")
print(f"P(Not sports) = {model.class_priors.get('Not sports', 0):.4f}")
print(f"\nWord probabilities:")
for w in words:
    if w in model.vocab:
        print(f"  P('{w}'|Sports)     = {model.word_probs['Sports'][w]:.4f}")
        print(f"  P('{w}'|Not sports) = {model.word_probs['Not sports'][w]:.4f}")
    else:
        print(f"  '{w}' is unknown (not in vocabulary)")

------------------------------------------------------------
NAIVE BAYES — SPORTS TEXT CLASSIFIER
                                                            

Test sentence: 'A very close game'

Predicted tag: Sports

Scores (log):
  Sports: -10.4960
  Not sports: -12.0720

--- Evaluation on Training Data ---
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-Score : 1.0000

--- Manual Calculation for 'A very close game' ---
Tokens: ['a', 'very', 'close', 'game']

P(Sports)     = 0.6000
P(Not sports) = 0.4000

Word probabilities:
  P('a'|Sports)     = 0.1200
  P('a'|Not sports) = 0.0870
  P('very'|Sports)     = 0.0800
  P('very'|Not sports) = 0.0435
  P('close'|Sports)     = 0.0400
  P('close'|Not sports) = 0.0870
  P('game'|Sports)     = 0.1200
  P('game'|Not sports) = 0.0435
